In [15]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

# Function to extract MFCCs from audio files
def extract_mfccs(file_path):
    try:
        y, sr = librosa.load(file_path, duration=1)  # Load 1-second audio
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        return np.mean(mfccs.T, axis=0)  # Average MFCCs over time
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Data Loading and Feature Extraction
male_folder = "/home/feliciano/adult_lobsters"  # Replace with your actual male folder name if different
female_folder = "/home/feliciano/juvenile_lobsters" # Replace with your actual female folder name if different

data = []
labels = []

for filename in os.listdir(male_folder):
    file_path = os.path.join(male_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("adult")

for filename in os.listdir(female_folder):
    file_path = os.path.join(female_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("juvenile")

if not data:
    print("No audio files found in the specified folders.")
    exit()

X = np.array(data)
y = np.array(labels)

# Label Encoding and Data Scaling
le = LabelEncoder()
y = le.fit_transform(y)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# KNN Model
knn_model = KNeighborsClassifier()

# Model Training and Evaluation
start_time = time.time()
knn_model.fit(X_train, y_train)
end_time = time.time()
inference_time = (end_time - start_time) * 1000 #milliseconds

y_pred = knn_model.predict(X_test)
y_pred_proba = knn_model.predict_proba(X_test)[:, 1] # Probability of the positive class (assuming 'female' is 1)

# Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_pred_proba)

# Print Results
print("KNN Model Performance:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  Inference Time (ms): {inference_time:.4f}")
print("\nNote: MAP-50 is not directly applicable for binary classification.")

KNN Model Performance:
  Accuracy: 0.9795
  Precision: 0.9583
  Recall: 0.9817
  F1-Score: 0.9698
  AUC-ROC: 0.9953
  Inference Time (ms): 0.8712

Note: MAP-50 is not directly applicable for binary classification.
